In [12]:
from dotenv import load_dotenv
load_dotenv()

True

In [14]:
# step 1 - load data 
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://en.wikipedia.org/wiki/Islam")
data= loader.load()


USER_AGENT environment variable not set, consider setting it to identify your requests.


In [ ]:
# step 2- split data into chunks

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000 , chunk_overlap=30)
documents =text_splitter.split_documents(data)

In [16]:
# step 3-  Converting to vectors 

from langchain_openai import OpenAIEmbeddings

embeddings= OpenAIEmbeddings()


In [17]:
# Storing in the vector db

from langchain_community.vectorstores import FAISS

vectorstoredb= FAISS.from_documents(documents , embeddings)
vectorstoredb

In [ ]:
# Querying from the vector db

query = "What are the principples of islam?"
result=vectorstoredb.similarity_search(query)
result[0].page_content

In [3]:
from langchain_openai import ChatOpenAI

llm= ChatOpenAI(model="gpt-4o")

In [18]:
# Retrival chain 
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

prompt = ChatPromptTemplate.from_template(
    """" 
     Answer the following question based on the provided context : 
      <context> {context} </context>"""
)

In [ ]:
# Document chain

document_chain= create_stuff_documents_chain(llm, prompt)

In [20]:
from langchain_core.documents import Document

document_chain.invoke({
    "input" : "Principles of islam?",
    "context" : [Document(page_content="Waht are the Principles of islam?")]
})

"The principles of Islam are primarily encapsulated in the Five Pillars, which are fundamental acts of worship and the foundation of a Muslim's faith and practice. These are:\n\n1. **Shahada (Faith)**: The declaration of faith, professing that there is no god but Allah, and Muhammad is His messenger. This proclamation affirms the monotheistic essence of Islam.\n\n2. **Salah (Prayer)**: Performing the five daily prayers at prescribed times throughout the day. These prayers are a direct link between the worshiper and Allah.\n\n3. **Zakat (Almsgiving)**: The giving of a fixed portion of one's wealth to the needy, usually calculated as 2.5% of one's savings. It purifies wealth and encourages economic equality.\n\n4. **Sawm (Fasting during Ramadan)**: Refraining from food, drink, and other physical needs during daylight hours in the month of Ramadan. This practice promotes self-discipline, spiritual growth, and empathy for those less fortunate.\n\n5. **Hajj (Pilgrimage to Mecca)**: A pilgri

In [24]:
# Retriver 

retriever = vectorstoredb.as_retriever()

from langchain_classic.chains import create_retrieval_chain

retrival_chain = create_retrieval_chain(retriever , document_chain)

response=retrival_chain.invoke({"input" : "Summarize the whole content of the page"})
response

{'input': 'Summarize the whole content of the page',
 'context': [Document(id='4856f8b7-8aa5-42bf-9b4e-7266c29e8579', metadata={'source': 'https://en.wikipedia.org/wiki/Islam', 'title': 'Islam - Wikipedia', 'language': 'en'}, page_content='This page was last edited on 30 April 2026, at 03:48\xa0(UTC).\nText is available under the Creative Commons Attribution-ShareAlike 4.0 License;\nadditional terms may apply. By using this site, you agree to the Terms of Use and Privacy Policy. Wikipedia® is a registered trademark of the Wikimedia Foundation, Inc., a non-profit organization.\n\n\nPrivacy policy\nAbout Wikipedia\nDisclaimers\nContact Wikipedia\nLegal & safety contacts\nCode of Conduct\nDevelopers\nStatistics\nCookie statement\nMobile view\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nSearch\n\n\n\n\n\n\n\n\n\n\n\n\n\nSearch\n\n\n\n\n\n\n\n\n\nToggle the table of contents\n\n\n\n\n\n\n\nIslam\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n268 languages\n\n\nAdd topic'),
  Document(id='644d8c2e